# Generate and Save Cost Hessian Data

Computes the leading eigenvalues/vectors of the loss Hessian w.r.t. stiffnesses K
for all trained networks in a specified data directory.

Results are saved alongside the network files as `cost_hessian.npz`.

**Warning:** each network takes several minutes (one JAX gradient + ~k Lanczos HVPs).
For large ensembles, use `training/runners/submit_cost_hessian.sh` instead.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('../../../').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Project root:', REPO_ROOT)

## Parameters

In [ ]:
# ── Network type ─────────────────────────────────────────────────────────────
NETWORK_TYPE = 'auxetic'   # 'auxetic' (allosteric not yet supported here)

# ── Data directory ───────────────────────────────────────────────────────────
DATA_ROOT  = REPO_ROOT / 'data'
SUBFOLDER  = 'targeted_results_sqr'
DATA_DIR   = DATA_ROOT / 'auxetic_nets' / SUBFOLDER

# ── Cost Hessian parameters ──────────────────────────────────────────────────
K_EIGS        = None    # None → all eigenvalues (n_edges - 1); set int to limit
HVP_EPSILON   = 1e-4
FORCE_TYPE    = 'quadratic'
N_STRAIN_STEPS = None   # None → use config.get_n_strain_steps(task_seed)
OVERWRITE     = False
VERBOSE       = True

# ── Task / realization filter ────────────────────────────────────────────────
TASK_FILTER   = None   # e.g. [0, 1, 2]
REAL_FILTER   = None

print(f'DATA_DIR: {DATA_DIR}  exists={DATA_DIR.exists()}')

In [ ]:
import numpy as np
from tqdm.notebook import tqdm

from analysis.data_io import load_auxetic_network
from analysis.cost_utils import compute_cost_hessian
from base.config import get_n_strain_steps
from training.src.task_generator import generate_task_config

print('Imports OK')

## Discover networks

In [ ]:
pairs = []
for task_path in sorted(DATA_DIR.glob('task_*')):
    task_seed = int(task_path.name.split('_')[1])
    if TASK_FILTER is not None and task_seed not in TASK_FILTER:
        continue
    for real_path in sorted(task_path.glob('realization_*')):
        real_seed = int(real_path.name.split('_')[1])
        if REAL_FILTER is not None and real_seed not in REAL_FILTER:
            continue
        if (real_path / 'final_network.pkl').exists():
            pairs.append((task_seed, real_seed))

print(f'Found {len(pairs)} networks')

## Compute and save cost Hessians

In [ ]:
errors = []

for task_seed, real_seed in tqdm(pairs, desc='Networks'):
    path = DATA_DIR / f'task_{task_seed:02d}' / f'realization_{real_seed:02d}'
    out  = path / 'cost_hessian.npz'

    if out.exists() and not OVERWRITE:
        print(f'  Skip task={task_seed} real={real_seed}: already exists')
        continue

    try:
        network, boundary = load_auxetic_network(task_seed, real_seed, DATA_DIR)
    except Exception as e:
        errors.append((task_seed, real_seed, str(e)))
        continue

    task_cfg = generate_task_config(task_seed)
    n_steps  = N_STRAIN_STEPS or get_n_strain_steps(task_seed)

    all_evals, all_evecs = [], []
    for subtask_idx, (cs, tp) in enumerate(
        zip(task_cfg['compression_strains'], task_cfg['target_poisson_ratios'])
    ):
        print(f'  task={task_seed} real={real_seed} subtask={subtask_idx}  cs={cs:.3f}  tp={tp:.3f}',
              flush=True)
        try:
            evals, evecs = compute_cost_hessian(
                network, cs, tp, boundary,
                k_eigs=K_EIGS, hvp_epsilon=HVP_EPSILON,
                force_type=FORCE_TYPE, n_strain_steps=n_steps, verbose=VERBOSE,
            )
            all_evals.append(evals)
            all_evecs.append(evecs)
        except Exception as e:
            errors.append((task_seed, real_seed, subtask_idx, str(e)))
            print(f'    ERROR: {e}')

    if all_evals:
        np.savez(
            out,
            eigenvalues=np.array(all_evals),   # (n_subtasks, k)
            eigenvectors=np.array(all_evecs),   # (n_subtasks, n_edges, k)
            compression_strains=np.array(task_cfg['compression_strains']),
            target_poissons=np.array(task_cfg['target_poisson_ratios']),
            task_seed=np.array(task_seed),
            realization_seed=np.array(real_seed),
        )
        print(f'  Saved → {out}')

print(f'\nDone. Errors: {len(errors)}')
if errors:
    for e in errors:
        print(' ', e)